In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [2]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)

def makesetflex (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    M = len(B)
    added = []
    for i in range(M):
        new = B[0:i+1]
        for k in range(N):
            if len(A[k])==len(new) and len(np.intersect1d(A[k],new))==len(new):
                break
            if k == N-1:
                A.append(new)
                added.append(new)
    return(A,added)

def robust_counterpart (sets,p,R,r,m,r_f,c):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        constraints.append((-R @ a)[i] - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    constraints.append(cp.abs(a)<= 10)
    #constraints.append(a>=0)
    #constraints.append(cp.sum(a)<=1)
    constraints.append(alpha + beta + gamma * (r-1) - (1-cp.sum(a))*r_f + z4 + z2 <= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)
    
def robustcheck(a,R,r,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value - (1-np.sum(a))*r_f,q_b.value)

In [8]:
def squeeze_algo(R,r,c,p,m,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=10]
    h = np.zeros(N)
    iterations = 1
    steps = 1
    f_obj = 0
    for i in range(N-1):
        h[i] = h_3(sum(p[i:N]),m)-h_3(sum(p[i+1:N]),m)
    h[N-1]=h_3(p[N-1],m)
    constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c+1e-5)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    oldrank = np.argsort(R.dot(w))
    sets = ranktoset(oldrank)
    nonstop = True
    while nonstop:
        [rbvalue,h] = robustcheck(w,R,r,p,m,r_f)
        print('rbvalue',rbvalue)
        if rbvalue <= c+1e-5:
            return('cut-stop',w,upperobj,iterations,steps)
        constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c+1e-5)
        iterations = iterations + 1
        nonstop2 = True
        once = False
        while nonstop2:
            [w,lowerobj] = robust_counterpart(sets,p,R,r,m,r_f,c)
            newrank = np.argsort(R.dot(w))
            if np.array_equal(newrank,oldrank):
                break
            once = True
            oldrank = newrank
            [sets,added] = makesetflex(sets, newrank)
            steps = steps + 1
            print('steps',steps)
        if upperobj - lowerobj <= 1e-10:
            return('gap stop', w,lowerobj,iterations,steps)
        if once:
            [rbvalue,h] = robustcheck(w,R,r,p,m,r_f)
            constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c+1e-5)
            iterations = iterations + 1
        #obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value 
        [sets,added] = makesetflex(sets, np.argsort(R.dot(w)))
        print(upperobj, lowerobj, iterations,steps)

In [19]:
np.random.seed(5)

In [20]:
N=30
p = np.zeros(N)+1/N
I = 3
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))
#print(R)

[0.07174797 0.07208475 0.06633768]


In [6]:
r = 0.3
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.001
c = 0.12

In [21]:
squeeze_algo(R,r,c,p,m,r_f)

rbvalue 2.8314805257029207
steps 2
0.15467574326003236 0.09302749571451872 3 2
rbvalue 4.408544459723747
steps 3
steps 4
0.09303510160534174 0.09302730459097798 5 4
rbvalue 0.12000999308251889


('cut-stop',
 array([0.45238081, 0.38079094, 0.50448148]),
 0.09303510160534174,
 5,
 4)